> **Публичная версия:** встроенные изображения результатов удалены из-за непроверенных прав на публикацию микрофотографий. Код, метрики и сохранённые численные результаты доступны без повторного обучения. Полные визуальные результаты сохраняются в отдельном архиве и могут предоставляться преподавателю по разрешённому каналу.

# Детекция и сегментация планктонных организмов на неразмеченных изображениях

**Лабораторная работа · максимум 50 баллов · воспроизводимое сравнение в Google Colab**

**Модели:** YOLO26n (one-stage), Faster R-CNN ResNet50-FPN V2 (two-stage), GrabCut и SAM 2.1 Hiera Tiny. Все веса предобучены; дообучение отсутствует.

**Важное ограничение:** веса YOLO/Faster R-CNN обучены на COCO и не имеют класса «планктон». Выходные метки не являются подтверждением таксономической принадлежности, а confidence — не оценка точности. При нуле детекций демонстрируется ручной box-prompt с явной отметкой источника.

**Структура:** анализ данных → аугментации → 4 группы экспериментов детекции → устойчивость боксов → 2 метода сегментации по одинаковым box prompts → таблицы/ошибки/выводы → экспорт результатов.

**Режим Colab:** Runtime → Change runtime type → T4 GPU. Код сохраняет результаты в `results/`, чтобы ревьюеру не требовалось запускать модели заново. Не публикуйте исходные изображения и производные иллюстрации без проверки разрешения правообладателя.

## 0. Архитектуры и аргументация

**YOLO26n — одноэтапная модель:** извлекает признаки с нескольких масштабов и напрямую предсказывает геометрию боксов и классы в одном проходе по изображению. Нет отдельного этапа proposal + ROI классификации. Компактный вариант `n` выбран, чтобы обеспечить относительно быстрые прогоны и исследовать перенос COCO-детектора на микроскопию.

**Faster R-CNN ResNet50-FPN V2 — двухэтапная модель:** backbone+FPN формируют многоуровневые признаки, RPN предлагает кандидаты, ROIAlign извлекает признаки для каждого региона, второй этап уточняет классы и координаты. Подход может быть более затратен, но позволяет отдельно уточнять предложенные регионы. Не следует ожидать гарантированного превосходства на планктоне без адаптации домена.

**GrabCut:** итеративно обучает модели цветов переднего плана и фона (GMM) и решает задачу минимального разреза графа, исходя из прямоугольника. При схожих цветах тела и фона, тонких щетинках и перекрытиях может ошибаться.

**SAM 2:** сначала вычисляет embedding изображения, затем декодирует маску по пространственной подсказке `xyxy`. Модель не требует специальной разметки планктона, но качество масок ограничено точностью подсказки и доменным сдвигом. Для сравнения используется тот же набор прямоугольников, что у GrabCut.

**Источники:** [Ultralytics YOLO](https://docs.ultralytics.com/modes/predict/) · [Torchvision Faster R-CNN](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn_v2.html) · [Meta SAM 2](https://github.com/facebookresearch/sam2) · [GrabCut paper](https://www.microsoft.com/en-us/research/publication/grabcut-interactive-foreground-extraction-using-iterated-graph-cuts/).

## 1. Установка зависимостей

Ячейка устанавливает инструменты непосредственно в Google Colab. PyTorch/torchvision в Colab уже установлены и не переустанавливаются без необходимости. Для SAM 2 отключена опциональная сборка CUDA-расширения (основные операции box-prompt inference остаются доступными). В иной среде зависимости ставятся по `requirements.txt` с совместимыми `torch/torchvision`.

In [1]:
import sys, subprocess, os
if 'google.colab' in sys.modules:
    env = os.environ.copy()
    env['SAM2_BUILD_CUDA'] = '0'
    packages = ['ultralytics>=8.3,<9', 'gdown>=6.0,<7', 'sam2>=1.1,<2',
                'opencv-python-headless>=4.8,<5', 'scipy>=1.11,<2',
                'pandas>=2,<3', 'matplotlib>=3.8,<4', 'Pillow>=10,<13',
                'huggingface-hub>=0.25,<2', 'scikit-image>=0.22,<1', 'tabulate>=0.9,<1']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages], env=env)
else:
    print('Локальная среда: установите requirements.txt, затем перезапустите kernel.')

## 2. Конфигурация, фиксированный seed и подключение датасета

Ссылка на папку: https://drive.google.com/drive/folders/1LT2NdDSULHiW3FwzT63ewgVO3pwRRhOW?usp=sharing

Универсальный сценарий: (1) загрузка публичной папки `gdown>=6`, (2) альтернативно скопировать изображения в `data/` вручную или подключить Drive. Поиск файлов рекурсивный, включая BMP, TIF/TIFF, JPG/PNG. Файлы с суффиксами `_obj_N_class_C_conf_S` считаются вторичными вырезками и исключаются из основного сравнения; классификация по имени носит эвристический характер.

Чтобы уложиться в бесплатный GPU, анализ охватывает **все найденные исходные файлы**, а детекция проводится на фиксированной **стратифицированной подвыборке** (`N_PER_GROUP=3`, обычно девять изображений). Для полного прогона увеличьте этот параметр.

In [2]:
from pathlib import Path
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import cv2, torch, json, re, time, random, hashlib, platform
from collections import Counter
from contextlib import nullcontext
from scipy.optimize import linear_sum_assignment
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path('data')
RESULTS = Path('results')
FOLDER_URL = 'https://drive.google.com/drive/folders/1LT2NdDSULHiW3FwzT63ewgVO3pwRRhOW'
AUTO_DOWNLOAD = True      # False для уже смонтированной/скопированной папки data/
N_PER_GROUP = 3         # ~9 разных исходных изображений для детекции
MAX_SIDE = 1536         # единственное изменение геометрии перед обеими моделями
YOLO_IMGSZ = 960
DET_CONF = 0.10         # exploratory threshold; не метрика качества
MAX_DET = 100
MAX_SEG_IMAGES = 6      # 6 изображений для сопоставления сегментации
MAX_PROMPTS_PER_IMAGE = 5
AUG_FOR_DETECTION = ['brightness_mild', 'noise_mild', 'blur_mild', 'flip_horizontal']

for path in ['analysis','augmentations','detection/yolo/original','detection/yolo/augmented',
             'detection/faster_rcnn/original','detection/faster_rcnn/augmented',
             'segmentation/grabcut/masks','segmentation/grabcut/overlays',
             'segmentation/sam2/masks','segmentation/sam2/overlays','segmentation/comparison',
             'metrics']:
    (RESULTS/path).mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, 'torch:',torch.__version__, 'torchvision:')
import torchvision
print(torchvision.__version__)

Device: cpu torch: 2.14.1+cpu torchvision:
0.29.1+cpu


In [3]:
EXT = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
DERIVED_PATTERN = re.compile(r'_obj_\d+_class_\d+_conf_', flags=re.I)

def enumerate_images(root):
    return sorted([p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in EXT], key=lambda p: str(p).lower())

if not enumerate_images(DATA_ROOT) and AUTO_DOWNLOAD:
    import gdown
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print('Загрузка Google Drive (интернет обязателен).')
    try:
        gdown.download_folder(url=FOLDER_URL, output=str(DATA_ROOT), quiet=False)
    except Exception as e:
        print('Автозагрузка не удалась:', repr(e))
        print('Альтернатива: скачайте папку ZIP, распакуйте в data/ или смонтируйте Drive и настройте DATA_ROOT.')

all_files = enumerate_images(DATA_ROOT)
raw_files = [p for p in all_files if not DERIVED_PATTERN.search(p.stem)]
derived_files = [p for p in all_files if DERIVED_PATTERN.search(p.stem)]
print(f'Всего файлов: {len(all_files)}, кандидатов исходных: {len(raw_files)}, производных: {len(derived_files)}')
assert raw_files, 'Исходные изображения не обнаружены. Проверьте DATA_ROOT/доступ к папке.'

def image_id(path):
    rel = str(path.relative_to(DATA_ROOT))
    stem = re.sub('[^a-zA-Z0-9]+', '_', path.stem)[:40].strip('_') or 'img'
    return f'{stem}_{hashlib.sha1(rel.encode()).hexdigest()[:8]}'

def open_rgb(path):
    with Image.open(path) as im:
        return np.array(ImageOps.exif_transpose(im).convert('RGB'))

def resize_working(im):
    h,w = im.shape[:2]
    s = min(1.0, MAX_SIDE/max(h,w))
    if s == 1: return im.copy(), (1.,1.)
    target_w = max(1, round(w*s)); target_h = max(1, round(h*s))
    return cv2.resize(im, (target_w,target_h), interpolation=cv2.INTER_AREA), (target_w/w, target_h/h)

def group_name(path):
    name=path.stem.lower()
    if 'bosmina' in name: return 'bosmina'
    if name.startswith('aakm'): return 'aakm'
    if re.match(r'^20\d{8,}', name): return 'timestamp'
    return 'other'

groups={g:[p for p in raw_files if group_name(p)==g] for g in sorted({group_name(p) for p in raw_files})}
# Include verified visual examples in the deterministic stratified sample, without annotations.
ANCHORS={
    'bosmina':['Bosmina cf longirostris Sackhalin (85).jpg',
               'Bosmina cf longirostris Sackhalin (84).jpg'],
    'aakm':['AAKM880(11).jpg'],
    'timestamp':['20250213140249905.jpg']}
selected=[]
for g,items in groups.items():
    pinned=[p for p in items if p.name in ANCHORS.get(g,[])]
    rest=[p for p in items if p not in pinned]
    rng=np.random.default_rng(SEED)
    wanted=max(0,N_PER_GROUP-len(pinned))
    idx=rng.choice(len(rest), size=min(wanted,len(rest)), replace=False)
    selected.extend(pinned+[rest[int(i)] for i in sorted(idx)])
selected=sorted(selected, key=lambda p: (group_name(p),p.name))
print('Выборка:', Counter(group_name(p) for p in selected))
print([p.name for p in selected])
(RESULTS/'metrics'/'selection.json').write_text(json.dumps({
    'seed':SEED, 'n_per_group':N_PER_GROUP, 'files':[str(p.relative_to(DATA_ROOT)) for p in selected],
    'derived_exclusion_regex':DERIVED_PATTERN.pattern,
    'dataset_url':FOLDER_URL},ensure_ascii=False, indent=2),encoding='utf-8')

Загрузка Google Drive (интернет обязателен).


Retrieving folder contents


Processing file 1SW7oTEnJJto30Zyx7tjveRqfj2LVqzih 0005.jpg
Processing file 106StVlK0nBzKrDI-u3C64XZPlHY53RZq 20250213131616824.jpg
Processing file 1rii_DF2mFLG2Xbw_AS5Tu9GdP3gdCkFJ 20250213131626559.jpg
Processing file 16y8KqiuABCOJ8m3820WbOWXhTbXU0htF 20250213131634306.jpg
Processing file 1yNq7SXwY5m8_Zc2-CP0oK_SoVAAoqGcf 20250213131652198.jpg
Processing file 13ejGH0s11TP4jZG6yyMloxNLY0r06_Th 20250213131707578.jpg
Processing file 1M3cj9iES3QcBQKJ4sBth7Cib8JpLK9_B 20250213131717749.jpg
Processing file 1XyxdQUnT3My2U6f4YgdFl5jebCiIZL1q 20250213131733581.jpg
Processing file 1nkwCvp-Q4zOW3ljR3m_nW4v-pHb_C5Pq 20250213131756895.jpg
Processing file 17rRWaRLfQgqJHWqEQzzf5Rp-30VCZ7mR 20250213131804115.jpg
Processing file 1Y-lSc-YRXkeW4KYkcjxe1H8i-dN3_QN9 20250213131814808.jpg
Processing file 1zS7Ct7AT6ZV8LL5TdqbrHCkAAX56-YW2 20250213131835294.jpg
Processing file 19j01d73zbOpvBrWy_HxcCPd8sq-tu7jn 20250213131844327.jpg
Processing file 1P1jyV7jGhjuRo5oVxZgqB5jCEvGhBW83 20250213131900592.jpg
Proce

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1SW7oTEnJJto30Zyx7tjveRqfj2LVqzih
To: /content/data/0005.jpg
100%|██████████| 1.94M/1.94M [00:00<00:00, 40.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=106StVlK0nBzKrDI-u3C64XZPlHY53RZq
To: /content/data/20250213131616824.jpg
100%|██████████| 529k/529k [00:00<00:00, 22.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1rii_DF2mFLG2Xbw_AS5Tu9GdP3gdCkFJ
To: /content/data/20250213131626559.jpg
100%|██████████| 473k/473k [00:00<00:00, 41.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=16y8KqiuABCOJ8m3820WbOWXhTbXU0htF
To: /content/data/20250213131634306.jpg
100%|██████████| 416k/416k [00:00<00:00, 40.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yNq7SXwY5m8_Zc2-CP0oK_SoVAAoqGcf
To: /content/data/20250213131652198.jpg
100%|██████████| 452k/452k [00:00<00:00, 14.6MB/s]
Downloading...
From: 

Автозагрузка не удалась: DownloadError('Failed to download the following files:\n- data/Bosmina cf longirostris Sackhalin (63)_obj_0_class_55_conf_0.18.jpg\n- data/Bosmina cf longirostris Sackhalin (64)_obj_0_class_0_conf_0.43.jpg\n- data/Bosmina cf longirostris Sackhalin (65)_obj_1_class_75_conf_0.16.jpg\n- data/Bosmina cf longirostris Sackhalin (7)_obj_2_class_58_conf_0.11.jpg\n- data/Bosmina cf longirostris Sackhalin (71)_obj_0_class_77_conf_0.25.jpg\n- data/Bosmina cf longirostris Sackhalin (72)_obj_0_class_0_conf_0.26.jpg\n- data/Bosmina cf longirostris Sackhalin (77)_obj_0_class_0_conf_0.14.jpg\n- data/Bosmina cf longirostris Sackhalin (78)_obj_0_class_54_conf_0.18.jpg\n- data/Bosmina cf longirostris Sackhalin (79)_obj_0_class_55_conf_0.31.jpg\n- data/Bosmina cf longirostris Sackhalin (83).jpg\n- data/Bosmina cf longirostris Sackhalin (84).jpg\n- data/Bosmina cf longirostris Sackhalin (85).jpg\n- data/Bosmina cf longirostris Sackhalin (89)_obj_0_class_0_conf_0.24.jpg\n- data/Bosm

Failed to download data/полифем4.jpg: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1sbMaTOEyMPfiOyCI72pVlxY9o-WVfM0N

but Gdown can't. Please check connections and permissions.


569

## Задание 1. Анализ исходных данных (2 балла)

Оценки яркости/контраста приведены как **характеристики изображения**, а не качества модели. Контраст = стандартное отклонение полутоновых уровней; энтропия оценивает текстурную неоднородность. Доля яркого фона — грубая эвристика, не истинная разметка фона. Перекрытия и морфология оцениваются визуально на нескольких снимках.

In [4]:
from skimage.measure import shannon_entropy
profile=[]
for p in raw_files:
    try:
        im=open_rgb(p); gray=cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
        profile.append({'file':str(p.relative_to(DATA_ROOT)), 'group':group_name(p),
                        'width_px':im.shape[1],'height_px':im.shape[0],
                        'brightness_mean':round(float(gray.mean()),2),
                        'contrast_std':round(float(gray.std()),2),
                        'entropy_bits':round(float(shannon_entropy(gray)),3),
                        'bright_pixel_fraction':round(float((gray>225).mean()),3)})
    except Exception as e: print('Не удалось прочитать',p,e)
profile_df=pd.DataFrame(profile)
profile_df.to_csv(RESULTS/'metrics'/'image_profile.csv',index=False)
print('Изображений прочитано:',len(profile_df)); display(profile_df.groupby('group').agg(
    n=('file','size'), min_width=('width_px','min'),max_width=('width_px','max'),
    min_height=('height_px','min'),max_height=('height_px','max'),
    mean_contrast=('contrast_std','mean')).round(2))

Изображений прочитано: 86


            n  min_width  max_width  min_height  max_height  mean_contrast
group                                                                     
aakm        7       1609       2124        1295        2000          78.00
bosmina     3       2071       2592        1620        1944          43.09
other      11       1276       3584        1168        2748          36.00
timestamp  65       2048       2048        1536        1536          24.21

In [5]:
fig, axes=plt.subplots(1,min(4,len(selected)),figsize=(17,5))
axes=np.atleast_1d(axes)
for ax,p in zip(axes,selected[:4]):
    ax.imshow(open_rgb(p)); ax.set_title(p.name[:35],fontsize=9);ax.axis('off')
plt.tight_layout();plt.savefig(RESULTS/'analysis'/'sample_gallery.png',dpi=135,bbox_inches='tight');plt.show()

fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].hist(profile_df['contrast_std'],bins=18);ax[0].set(title='Распределение контраста',xlabel='std(gray)')
ax[1].scatter(profile_df['brightness_mean'],profile_df['contrast_std'],s=18,alpha=.65)
ax[1].set(title='Яркость / контраст',xlabel='mean(gray)',ylabel='std(gray)')
plt.tight_layout();plt.savefig(RESULTS/'analysis'/'dataset_statistics.png',dpi=150);plt.show()

<Figure size 1700x500 with 4 Axes>

<Figure size 1200x400 with 2 Axes>

### Наблюдения по исходным изображениям

По визуально изученным примерам (до запуска ноутбука) фон меняется от белого до серого; тонкие прозрачные отростки могут почти исчезать на светлом фоне. На цветных кадрах с множеством особей заметны контактирующие тела и частичное перекрытие. Изображения с плотными тёмными областями и зернистым фоном создают риск ложных границ, особенно для GrabCut. Итоговые численные диапазоны размеров, яркости и контраста выводятся кодом выше, поэтому **их нельзя заменять числами без фактического запуска**.

## Задание 1. Аугментации и анализ (8 баллов)

Применяются как минимум четыре класса преобразований: яркость/контраст, гауссов шум, гауссово размытие, геометрическое отражение. Дополнительно — гамма-коррекция и поворот. Визуализируются несколько уровней каждого фотометрического преобразования. Для геометрической аугментации храним обратное преобразование боксов; измерять IoU в разных системах координат было бы ошибкой.

In [6]:
AUG_PARAMS={
    'original': {},
    'brightness_mild': {'alpha':1.10,'beta':10},
    'brightness_strong': {'alpha':1.40,'beta':25},
    'contrast_low': {'alpha':0.85,'beta':0},
    'contrast_high': {'alpha':1.25,'beta':0},
    'noise_mild': {'sigma':8},
    'noise_strong': {'sigma':25},
    'blur_mild': {'kernel':3},
    'blur_strong': {'kernel':7},
    'gamma_low': {'gamma':0.75},
    'gamma_high': {'gamma':1.40},
    'flip_horizontal': {},
    'rotate90': {},
}

def stable_seed(key,aug_name):
    return (int(hashlib.sha1(f'{key}|{aug_name}|{SEED}'.encode()).hexdigest()[:8],16))

def augment(img, name, key='img'):
    if name=='original': return img.copy()
    a = img.astype(np.float32)
    if name.startswith('brightness') or name.startswith('contrast'):
        v=AUG_PARAMS[name]; return np.clip(a*v['alpha']+v['beta'],0,255).astype(np.uint8)
    if name.startswith('noise'):
        noise=np.random.default_rng(stable_seed(key,name)).normal(0,AUG_PARAMS[name]['sigma'],img.shape)
        return np.clip(a+noise,0,255).astype(np.uint8)
    if name.startswith('blur'):
        k=AUG_PARAMS[name]['kernel']; return cv2.GaussianBlur(img,(k,k),0)
    if name.startswith('gamma'):
        gamma=AUG_PARAMS[name]['gamma'];return np.clip(255*(a/255)**gamma,0,255).astype(np.uint8)
    if name=='flip_horizontal': return cv2.flip(img,1)
    if name=='rotate90': return cv2.rotate(img,cv2.ROTATE_90_CLOCKWISE)
    raise ValueError(name)

def inverse_box(box, aug_name, original_shape):
    # Coordinate convention: xyxy using continuous image boundaries [0,w] and [0,h].
    x1,y1,x2,y2=map(float,box);h,w=original_shape[:2]
    if aug_name=='flip_horizontal': return [w-x2,y1,w-x1,y2]
    if aug_name=='rotate90': return [y1,h-x2,y2,h-x1]
    return [x1,y1,x2,y2]

sample_path=selected[0]; sample,_=resize_working(open_rgb(sample_path))
names=['original','brightness_mild','brightness_strong','contrast_low','contrast_high',
       'noise_mild','noise_strong','blur_mild','blur_strong',
       'gamma_low','gamma_high','flip_horizontal','rotate90']
fig,ax=plt.subplots(4,4,figsize=(15,13));
for j,(a,name) in enumerate(zip(ax.flat,names)):
    result=augment(sample,name,key=image_id(sample_path));a.imshow(result);a.set_title(name,fontsize=10);a.axis('off')
for a in ax.flat[len(names):]:a.axis('off')
plt.tight_layout();plt.savefig(RESULTS/'augmentations'/'levels_gallery.png',dpi=130);plt.show()
(RESULTS/'metrics'/'augmentation_params.json').write_text(json.dumps(AUG_PARAMS,indent=2),encoding='utf-8')

<Figure size 1500x1300 with 16 Axes>

541

### Морфологическая обоснованность аугментаций

- **Умеренная яркость и контраст** имитируют изменения освещения/экспозиции. Сильное высветление вызывает клиппинг прозрачных границ и потерю информации; чрезмерный контраст может «запечатать» внутренние структуры.
- **Слабый шум** полезен как имитация сенсора; высокий шум маскирует щетинки, тонкие усики и мелкие детали, увеличивая риск ложных детекций.
- **Размытие 3×3** приближает дефокус микроскопа, но 7×7 и сильнее удаляет тонкие анатомические структуры, поэтому должно рассматриваться как стресс-тест, а не обязательная аугментация обучения.
- **Повороты/отражения** допустимы, когда ориентация организма не является целевым признаком, но могут отсекать объект на границах и менять видимость перекрытий. В задачах, где ориентация имеет биологический смысл, эти преобразования стоит ограничивать.
- **Гамма** влияет на слабоконтрастные прозрачные области нелинейно: умеренная коррекция допустима, экстремальная приводит к потере деталей.

## Задание 2. Загрузка предобученных YOLO и Faster R-CNN

Одинаковые входные RGB-массивы и порог `DET_CONF`, фиксированный `MAX_SIDE`. У моделей разные COCO-классификаторы и внутренний постпроцессинг, поэтому сравнение confidence между моделями **не калибровано**. Время измеряется как wall-clock для всего предсказания с синхронизацией CUDA; прогрев исключён.

In [7]:
from ultralytics import YOLO
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights)

yolo = YOLO('yolo26n.pt')
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
faster = fasterrcnn_resnet50_fpn_v2(weights=weights).to(DEVICE).eval()
faster_preprocess = weights.transforms()
COCO_CATEGORIES = weights.meta['categories']

def synchronize():
    if DEVICE=='cuda': torch.cuda.synchronize()

def detect_yolo(image):
    synchronize();t0=time.perf_counter()
    result=yolo.predict(source=cv2.cvtColor(image,cv2.COLOR_RGB2BGR), imgsz=YOLO_IMGSZ,conf=DET_CONF,
                        max_det=MAX_DET,device=0 if DEVICE=='cuda' else 'cpu',
                        verbose=False)[0]
    synchronize();elapsed=time.perf_counter()-t0
    det=[]
    if result.boxes is not None:
        for b,s,c in zip(result.boxes.xyxy.cpu().numpy(),result.boxes.conf.cpu().numpy(),result.boxes.cls.cpu().numpy()):
            det.append({'box':[float(v) for v in b], 'score':float(s),
                        'label':str(result.names[int(c)]), 'class_id':int(c)})
    return det,elapsed

@torch.inference_mode()
def detect_faster(image):
    synchronize(); t0=time.perf_counter()
    tensor=faster_preprocess(Image.fromarray(image)).to(DEVICE)
    pred=faster([tensor])[0]
    synchronize();elapsed=time.perf_counter()-t0
    det=[]
    for b,s,c in zip(pred['boxes'].cpu().numpy(),pred['scores'].cpu().numpy(),pred['labels'].cpu().numpy()):
        if float(s) < DET_CONF: continue
        label=COCO_CATEGORIES[int(c)] if int(c)<len(COCO_CATEGORIES) else str(int(c))
        det.append({'box':[float(v) for v in b], 'score':float(s),
                    'label':label,'class_id':int(c)})
        if len(det)>=MAX_DET: break
    return det,elapsed

DETECTORS={'yolo': detect_yolo,'faster_rcnn':detect_faster}
# Прогрев обеих моделей на одной и той же форме тестовой картинки (вне таймингов).
warm=np.full((384,384,3),127,dtype=np.uint8)
for model_name,detector in DETECTORS.items():
    try:
        _, warm_t = detector(warm)
        print(f'{model_name} прогрет: {warm_t:.3f} с (не включается в таблицы)')
    except Exception as e:
        raise RuntimeError(f'Не удалось запустить {model_name}; проверьте пакеты и runtime: {e}')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:01<00:00, 118MB/s]


yolo прогрет: 0.643 с (не включается в таблицы)
faster_rcnn прогрет: 13.254 с (не включается в таблицы)


## Задание 2. Четыре группы экспериментов (4×)

По каждой исходной картинке выполняются оригинал + **четыре сопоставимые слабые аугментации** на каждой модели. Результаты детекции фиксируются в JSONL и CSV, а исходные/аугментированные кадры с боксами — в PNG. Это независимые инференсы, а не перенос предсказаний исходного изображения.

In [8]:
import matplotlib.patches as patches

def draw_boxes(img,dets,heading,save_path):
    fig,ax=plt.subplots(figsize=(10,7));ax.imshow(img)
    for d in dets:
        x1,y1,x2,y2=d['box'];rect=patches.Rectangle((x1,y1),max(1,x2-x1),max(1,y2-y1),
                     fill=False,edgecolor='lime',linewidth=1.6)
        ax.add_patch(rect)
        ax.text(x1,max(0,y1-3),f"{d['label']} {d['score']:.2f}",color='black',fontsize=7,
                bbox={'facecolor':'lime','alpha':0.72,'pad':1,'edgecolor':'none'})
    ax.set_title(f'{heading} | n={len(dets)}');ax.axis('off')
    fig.savefig(save_path,dpi=115,bbox_inches='tight');plt.close(fig)

observations=[]; detection_rows=[]; original_dets={}; images_work={}
for i,p in enumerate(selected,1):
    rgb_original=open_rgb(p)
    image,scale_xy=resize_working(rgb_original)
    iid=image_id(p)
    images_work[iid] = image
    print(f'[{i}/{len(selected)}] {p.name} ({image.shape[1]}×{image.shape[0]})')
    for method,runner in DETECTORS.items():
        for variant in ['original']+AUG_FOR_DETECTION:
            view=augment(image,variant,key=iid)
            dets,dt=runner(view)
            experiment=1 if method=='yolo' and variant=='original' else \
                       2 if method=='yolo' else 3 if variant=='original' else 4
            key=(iid,method,variant)
            observations.append({'image_id':iid,'source_file':str(p.relative_to(DATA_ROOT)),
                'model':method,'variant':variant,'experiment':experiment,
                'width_original':rgb_original.shape[1], 'height_original':rgb_original.shape[0],
                'width_work':image.shape[1], 'height_work':image.shape[0],
                'scale_x':scale_xy[0],'scale_y':scale_xy[1],
                'count':len(dets),'inference_seconds':float(dt),'detections':dets})
            if variant=='original':original_dets[(iid,method)]=dets
            for j,d in enumerate(dets):
                box_back=inverse_box(d['box'],variant,image.shape)
                box_orig=[box_back[0]/scale_xy[0],box_back[1]/scale_xy[1],
                          box_back[2]/scale_xy[0],box_back[3]/scale_xy[1]]
                detection_rows.append({'image_id':iid,'source_file':p.name,'experiment':experiment,
                    'model':method,'variant':variant,'det_id':j,'score':d['score'],
                    'label_coco':d['label'],'class_id_coco':d['class_id'],
                    'x1_variant':d['box'][0],'y1_variant':d['box'][1],
                    'x2_variant':d['box'][2],'y2_variant':d['box'][3],
                    'x1_work':box_back[0],'y1_work':box_back[1],
                    'x2_work':box_back[2],'y2_work':box_back[3],
                    'x1_original':box_orig[0],'y1_original':box_orig[1],
                    'x2_original':box_orig[2],'y2_original':box_orig[3],
                    'time_image_seconds':dt})
            folder=f'detection/{method}/original' if variant=='original' else f'detection/{method}/augmented'
            draw_boxes(view,dets,f'{method} / {variant}',RESULTS/folder/f'{iid}_{variant}.png')

with (RESULTS/'metrics'/'predictions.jsonl').open('w',encoding='utf-8') as f:
    for row in observations:f.write(json.dumps(row,ensure_ascii=False)+'\n')
DET_COLS=['image_id','source_file','experiment','model','variant','det_id','score','label_coco','class_id_coco',
          'x1_variant','y1_variant','x2_variant','y2_variant','x1_work','y1_work','x2_work','y2_work',
          'x1_original','y1_original','x2_original','y2_original','time_image_seconds']
pd.DataFrame(detection_rows,columns=DET_COLS).to_csv(RESULTS/'metrics'/'detections.csv',index=False)
image_level_df=pd.DataFrame([{k:v for k,v in obs.items() if k!='detections'} for obs in observations])
image_level_df.to_csv(RESULTS/'metrics'/'detection_image_level.csv',index=False)
print('Инференсов:',len(observations),'обнаружений:',len(detection_rows))

[1/12] AAKM801(1).jpg (1536×1211)
[2/12] AAKM828(9).jpg (1536×1106)
[3/12] AAKM880(11).jpg (1536×1336)
[4/12] Bosmina cf longirostris Sackhalin (24).jpg (1536×1202)
[5/12] Bosmina cf longirostris Sackhalin (50).jpg (1536×1331)
[6/12] Bosmina cf longirostris Sackhalin (53).jpg (1536×1152)
[7/12] 0005.jpg (1536×1152)
[8/12] B afffinis.JPG (1536×1152)
[9/12] bosm long3.jpg (1536×1152)
[10/12] 20250213131717749.jpg (1536×1152)
[11/12] 20250213135744652.jpg (1536×1152)
[12/12] 20250213140249905.jpg (1536×1152)
Инференсов: 120 обнаружений: 713


## Задание 2. Анализ устойчивости и сравнение детекторов

Bounding boxes на аугментированных изображениях (при `flip_horizontal`) приводятся в систему координат исходного снимка. Сопоставление выполняется Венгерским алгоритмом по IoU. `match_rate` — доля базовых боксов, для которых найден бокс с IoU ≥ 0.30. Это **устойчивость геометрии предсказаний**, не Recall: без истинных размеченных объектов корректно измерять Recall невозможно. Когда базовых боксов нет, показатель пропускается (`NaN`), а не объявляется 100%.

In [9]:
def box_iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]);y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1]);area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(area_a+area_b-inter+1e-9)

def compare_boxes(baseline, other, variant, original_shape, iou_threshold=.30):
    if not baseline:return {'matched':0,'match_rate':np.nan,'mean_matched_iou':np.nan}
    if not other:return {'matched':0,'match_rate':0.,'mean_matched_iou':np.nan}
    bb=[d['box'] for d in baseline]
    bb_other=[inverse_box(d['box'],variant,original_shape) for d in other]
    mat=np.array([[box_iou(a,b) for b in bb_other] for a in bb])
    rr,cc=linear_sum_assignment(1-mat)
    ok=np.array([mat[r,c] for r,c in zip(rr,cc) if mat[r,c]>=iou_threshold])
    return {'matched':int(len(ok)),'match_rate':float(len(ok)/len(bb)),
            'mean_matched_iou':float(ok.mean()) if len(ok) else np.nan}

stability=[]
for obs in observations:
    if obs['variant']=='original':continue
    shape=images_work[obs['image_id']].shape
    baseline=original_dets[(obs['image_id'],obs['model'])]
    cmp=compare_boxes(baseline,obs['detections'],obs['variant'],shape)
    stability.append({'image_id':obs['image_id'],'model':obs['model'],'variant':obs['variant'],
        'base_count':len(baseline),'aug_count':obs['count'],
        'count_delta':obs['count']-len(baseline), **cmp})
stability_df=pd.DataFrame(stability)
stability_df.to_csv(RESULTS/'metrics'/'box_stability.csv',index=False)

summary=image_level_df.groupby(['experiment','model']).agg(
    n_inferences=('count','size'),mean_detections=('count','mean'),
    zero_detection_fraction=('count',lambda s:(s==0).mean()),
    mean_inference_seconds=('inference_seconds','mean'),
    median_inference_seconds=('inference_seconds','median')).reset_index()
summary=summary.merge(stability_df.groupby('model').agg(
    match_rate_mean=('match_rate','mean'),matched_iou_mean=('mean_matched_iou','mean')).reset_index(),
    on='model',how='left')
summary.loc[summary['experiment'].isin([1,3]),['match_rate_mean','matched_iou_mean']]=np.nan
summary.to_csv(RESULTS/'metrics'/'detection_comparison.csv',index=False)
display(summary.round(3))

fig,ax=plt.subplots(1,2,figsize=(12,4))
for name,grp in image_level_df.groupby('model'):
    means=grp.groupby('variant')['count'].mean().reindex(['original']+AUG_FOR_DETECTION)
    ax[0].plot(range(len(means)),means.values,marker='o',label=name)
ax[0].set_xticks(range(5),['orig','bright','noise','blur','flip'],rotation=30)
ax[0].set(title='Количество обнаружений / изображение',ylabel='шт. (не accuracy)');ax[0].legend()
for name,grp in image_level_df.groupby('model'):
    ax[1].bar(name,grp['inference_seconds'].mean())
ax[1].set(title='Среднее время одного инференса',ylabel='секунды')
plt.tight_layout();plt.savefig(RESULTS/'metrics'/'detector_comparison.png',dpi=150);plt.show()

   experiment        model  n_inferences  mean_detections  \
0           1         yolo            12            2.917   
1           2         yolo            48            2.688   
2           3  faster_rcnn            12            8.500   
3           4  faster_rcnn            48            9.312   

   zero_detection_fraction  mean_inference_seconds  median_inference_seconds  \
0                    0.083                   0.308                     0.281   
1                    0.229                   0.307                     0.280   
2                    0.000                  13.387                    12.868   
3                    0.000                  12.907                    12.780   

   match_rate_mean  matched_iou_mean  
0              NaN               NaN  
1            0.762             0.912  
2              NaN               NaN  
3            0.851             0.905  

<Figure size 1200x400 with 2 Axes>

### Интерпретация экспериментов

Детектор выбирается после просмотра PNG с учётом (1) доли изображений без обнаружений, (2) повторяемости положения боксов при слабых аугментациях, (3) визуально заметных ложных обнаружений/пропусков, (4) времени инференса. Число обнаружений **не означает число реальных организмов**. Если оба COCO-детектора плохо работают, это ожидаемый отрицательный результат трансфера, и используется явно помеченный ручной bounding box для сравнения методов сегментации.

## Задание 3. Подготовка одинаковых bounding boxes и ручной fallback

Для сегментации берём оригинальные (не аугментированные) боксы одного выбранного детектора. **Предпочтение** задаётся вручную в переменной `SEG_DETECTOR` после анализа таблиц: автоматически выбирать «лучшего» исключительно по числу срабатываний некорректно. Если боксов нет, используем ограниченный набор демонстрационных прямоугольников на известных снимках (их источник = `manual_fallback`), а не выдаём их за детекции модели.

Ручные рамки заданы в **долях ширины/высоты** и применяются только на конкретных названных файлах. Перед сравнением масок проверьте их визуально и при необходимости отредактируйте.

In [10]:
SEG_DETECTOR='yolo'  # после визуальной проверки можно сменить на 'faster_rcnn'
# Координаты x1,y1,x2,y2 / W,H — эвристические демонстрационные рамки,
# приблизительно выбранные по реальным примерам; НЕ ground truth.
MANUAL_BOXES_REL={
    'Bosmina cf longirostris Sackhalin (85).jpg': [[0.11,0.04,0.88,0.93]],
    'Bosmina cf longirostris Sackhalin (84).jpg': [[0.03,0.07,0.93,0.91]],
    'AAKM880(11).jpg': [[0.02,0.02,0.97,0.98]],
    '20250213140249905.jpg': [[0.41,0.17,0.61,0.41], [0.55,0.28,0.80,0.53]],
}

# Prefer known cases with different morphology/overlap in the segmentation sample.
SEG_ANCHOR_NAMES={name for names in ANCHORS.values() for name in names}
seg_order=([p for p in selected if p.name in SEG_ANCHOR_NAMES] +
           [p for p in selected if p.name not in SEG_ANCHOR_NAMES])
seg_cases=[]
for p in seg_order[:MAX_SEG_IMAGES]:
    iid=image_id(p);im=images_work[iid];h,w=im.shape[:2]
    candidates=original_dets[(iid,SEG_DETECTOR)]
    chosen=candidates[:MAX_PROMPTS_PER_IMAGE]
    source='detector'
    if not chosen and p.name in MANUAL_BOXES_REL:
        chosen=[{'box':[r[0]*w,r[1]*h,r[2]*w,r[3]*h],
                 'score':None,'label':'manual','class_id':-1} for r in MANUAL_BOXES_REL[p.name]]
        source='manual_fallback'
    if not chosen:
        print('Нет боксов и нет ручной рамки:',p.name,'(снимок пропущен)')
        continue
    for j,d in enumerate(chosen):
        seg_cases.append({'image_id':iid,'filename':p.name,'box_id':j,'source':source,
                          'detector':SEG_DETECTOR if source=='detector' else None,
                          'score':d['score'],'box':[float(v) for v in d['box']]})
print('Сегментационных подсказок:',len(seg_cases),'ручных:',sum(c['source']=='manual_fallback' for c in seg_cases))
(RESULTS/'metrics'/'segmentation_prompts.json').write_text(
    json.dumps(seg_cases,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')

Сегментационных подсказок: 11 ручных: 0


3473

## Задание 3. GrabCut: бинарные маски по прямоугольникам

Чтобы не допустить исходной ошибки сегментации, боксы слегка сужаются к краю изображения, а `cv2.grabCut` получает прямоугольник в формате `x,y,width,height`. Маски сохраняются в PNG 0/255. Если алгоритм падает (слишком узкий бокс/вырожденный foreground), результат помечается ошибкой, **не заменяется выдуманной маской**.

**Вычислительный бюджет:** перед GrabCut регион с полями фона уменьшается до длинной стороны не более 768 px, а бинарная маска возвращается в рабочее разрешение методом nearest-neighbor. Это отдельный параметр эксперимента (`GRABCUT_MAX_SIDE`), который может ухудшить тонкие детали; в полном отчёте сравните этот эффект с внутренним ресайзингом SAM 2.


In [11]:
def clamp_box(box,w,h,margin=2):
    x1,y1,x2,y2=box
    x1=int(np.clip(np.floor(x1),margin,w-margin-2));y1=int(np.clip(np.floor(y1),margin,h-margin-2))
    x2=int(np.clip(np.ceil(x2),x1+2,w-margin));y2=int(np.clip(np.ceil(y2),y1+2,h-margin))
    return x1,y1,x2,y2

GRABCUT_MAX_SIDE = 768  # documented performance/accuracy trade-off; SAM2 keeps its own internal scale.
def grabcut_predict(rgb,box,iterations=5):
    h,w=rgb.shape[:2]
    x1,y1,x2,y2=clamp_box(box,w,h)
    assert x2-x1>=2 and y2-y1>=2
    # Include background context outside the shared box, then work on a bounded-size crop.
    pad_x=max(12,int((x2-x1)*.20));pad_y=max(12,int((y2-y1)*.20))
    left=max(0,x1-pad_x);top=max(0,y1-pad_y)
    right=min(w,x2+pad_x);bottom=min(h,y2+pad_y)
    crop=rgb[top:bottom,left:right]
    ch,cw=crop.shape[:2]
    s=min(1.,GRABCUT_MAX_SIDE/max(ch,cw))
    view=cv2.resize(crop,(round(cw*s),round(ch*s)),interpolation=cv2.INTER_AREA) if s<1 else crop
    vh,vw=view.shape[:2]
    local=[(x1-left)*s,(y1-top)*s,(x2-left)*s,(y2-top)*s]
    rx1,ry1,rx2,ry2=clamp_box(local,vw,vh,margin=2)
    gc_mask=np.zeros((vh,vw),np.uint8)
    bgd=np.zeros((1,65),np.float64);fgd=np.zeros((1,65),np.float64)
    cv2.grabCut(cv2.cvtColor(view,cv2.COLOR_RGB2BGR).copy(),gc_mask,
                (rx1,ry1,rx2-rx1,ry2-ry1),bgd,fgd,iterations,cv2.GC_INIT_WITH_RECT)
    mask_small=np.isin(gc_mask,[cv2.GC_FGD,cv2.GC_PR_FGD]).astype(np.uint8)
    mask_crop=cv2.resize(mask_small,(cw,ch),interpolation=cv2.INTER_NEAREST) if s<1 else mask_small
    full=np.zeros((h,w),dtype=np.uint8)
    full[top:bottom,left:right]=mask_crop
    return full

def save_mask_and_overlay(img,mask,path_mask,path_overlay,box=None):
    cv2.imwrite(str(path_mask),mask.astype(np.uint8)*255)
    overlay=img.copy().astype(np.float32); colored=np.array([255,70,30],dtype=np.float32)
    overlay[mask.astype(bool)] = .62*overlay[mask.astype(bool)] + .38*colored
    ov=overlay.astype(np.uint8)
    if box is not None:
        x1,y1,x2,y2=clamp_box(box,img.shape[1],img.shape[0]);
        cv2.rectangle(ov,(x1,y1),(x2,y2),(0,255,0),2)
    Image.fromarray(ov).save(path_overlay)

grab_outputs={}; seg_results=[]
for c in seg_cases:
    iid,j=c['image_id'],c['box_id']; im=images_work[iid];start=time.perf_counter()
    try:
        mask=grabcut_predict(im,c['box']);elapsed=time.perf_counter()-start
        grab_outputs[(iid,j)]=mask
        save_mask_and_overlay(im,mask,
            RESULTS/'segmentation/grabcut/masks'/f'{iid}_{j}.png',
            RESULTS/'segmentation/grabcut/overlays'/f'{iid}_{j}.png',c['box'])
        seg_results.append({**{k:v for k,v in c.items() if k!='box'},'method':'GrabCut',
                            'seconds':elapsed,'mask_area_fraction':float(mask.mean()),
                            'success':True,'error':''})
    except Exception as e:
        seg_results.append({**{k:v for k,v in c.items() if k!='box'},'method':'GrabCut',
                            'seconds':time.perf_counter()-start,'mask_area_fraction':np.nan,
                            'success':False,'error':repr(e)})
print('GrabCut успешных:',len(grab_outputs),'из',len(seg_cases))

GrabCut успешных: 11 из 11


## Задание 3. SAM 2.1 по тем же рамкам

`set_image` вызывается **один раз на изображение**, затем `predict(box=..., multimask_output=False)` для каждого соответствующего бокса; время embedding учитывается один раз на изображение и **амортизируется** между его prompt-ами при сопоставлении с GrabCut. Маска сохраняется в исходном рабочем разрешении. CUDA autocast применяется только на GPU.

In [12]:
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.build_sam import build_sam2_hf
sam_model = build_sam2_hf('facebook/sam2.1-hiera-tiny',device=DEVICE)
sam = SAM2ImagePredictor(sam_model)
sam.model.eval()
AMP_DTYPE = torch.bfloat16 if DEVICE=='cuda' and torch.cuda.is_bf16_supported() else torch.float16
sam_outputs={}; sam_timing=[]
for iid in sorted({c['image_id'] for c in seg_cases}):
    cases=[c for c in seg_cases if c['image_id']==iid]
    img=images_work[iid]
    synchronize();t0=time.perf_counter()
    with torch.inference_mode():
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE) if DEVICE=='cuda' else nullcontext():
            sam.set_image(img)
    synchronize();embedding_seconds=time.perf_counter()-t0
    for c in cases:
        j=c['box_id'];start=time.perf_counter()
        try:
            with torch.inference_mode():
                with torch.autocast(device_type='cuda',dtype=AMP_DTYPE) if DEVICE=='cuda' else nullcontext():
                    masks,quality,_=sam.predict(box=np.array(c['box'],dtype=np.float32),multimask_output=False)
            synchronize();prompt_sec=time.perf_counter()-start
            mask=(np.asarray(masks[0])>0).astype(np.uint8)
            assert mask.shape==img.shape[:2], f'Размер SAM2 маски {mask.shape}, изображения {img.shape[:2]}'
            sam_outputs[(iid,j)]=mask
            save_mask_and_overlay(img,mask,
                RESULTS/'segmentation/sam2/masks'/f'{iid}_{j}.png',
                RESULTS/'segmentation/sam2/overlays'/f'{iid}_{j}.png',c['box'])
            total_amortized=prompt_sec+embedding_seconds/len(cases)
            seg_results.append({**{k:v for k,v in c.items() if k!='box'},'method':'SAM2',
                'seconds':total_amortized,'prompt_seconds':prompt_sec,
                'embedding_shared_seconds':embedding_seconds,
                'mask_area_fraction':float(mask.mean()),'sam_predicted_quality':float(np.asarray(quality).reshape(-1)[0]),
                'success':True,'error':''})
        except Exception as e:
            seg_results.append({**{k:v for k,v in c.items() if k!='box'},'method':'SAM2',
                'seconds':time.perf_counter()-start+embedding_seconds/len(cases),
                'success':False,'error':repr(e)})
    sam_timing.append({'image_id':iid,'embedding_seconds':embedding_seconds,'n_prompts':len(cases)})
print('SAM2 успешных:',len(sam_outputs),'из',len(seg_cases))

sam2.1_hiera_tiny.pt: reconstructing file:   0%|          |  0.00B /  156MB            

sam2.1_hiera_tiny.pt: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


SAM2 успешных: 11 из 11


## Задание 3. Визуализация масок и сравнение методов

Вспомогательные показатели: доля пикселей маски и IoU **между результатами двух методов**. Это показатель согласованности, а не IoU по эталону. Нельзя трактовать высокое согласие как высокую точность. Замеры времени SAM2: на маску приходится время prompt + равная доля embedding изображения. Выводятся и полные времена на изображение.

In [13]:
mask_agreement=[]
for c in seg_cases:
    key=(c['image_id'],c['box_id'])
    if key not in grab_outputs or key not in sam_outputs:continue
    m1=grab_outputs[key]>0;m2=sam_outputs[key]>0
    intersection=np.logical_and(m1,m2).sum();union=np.logical_or(m1,m2).sum()
    agreement=float(intersection/union) if union else np.nan
    mask_agreement.append({'image_id':key[0],'box_id':key[1],'iou_between_methods':agreement})
    im=images_work[key[0]]
    fig,ax=plt.subplots(1,4,figsize=(16,4))
    ax[0].imshow(im);ax[0].add_patch(patches.Rectangle(
        (c['box'][0],c['box'][1]),c['box'][2]-c['box'][0],c['box'][3]-c['box'][1],
        fill=False,edgecolor='lime',linewidth=2));ax[0].set_title('Input + shared box')
    ax[1].imshow(m1,cmap='gray');ax[1].set_title('GrabCut mask')
    ax[2].imshow(m2,cmap='gray');ax[2].set_title('SAM 2 mask')
    ax[3].imshow(im);ax[3].imshow(np.ma.masked_where(~m2,m2),alpha=.45,cmap='autumn')
    ax[3].set_title(f'SAM 2 overlay | inter-IoU={agreement:.3f}')
    for a in ax:a.axis('off')
    plt.tight_layout();plt.savefig(RESULTS/'segmentation/comparison'/f'{key[0]}_{key[1]}.png',dpi=125);plt.close(fig)

seg_df=pd.DataFrame(seg_results)
seg_df.to_csv(RESULTS/'metrics'/'segmentation_timing.csv',index=False)
pd.DataFrame(mask_agreement,columns=['image_id','box_id','iou_between_methods']).to_csv(
    RESULTS/'metrics'/'mask_agreement.csv',index=False)
pd.DataFrame(sam_timing).to_csv(RESULTS/'metrics'/'sam_embedding_timing.csv',index=False)
seg_summary=seg_df.groupby('method').agg(n_masks=('success','size'),success_rate=('success','mean'),
    mean_seconds_per_mask_amortized=('seconds','mean'),
    median_seconds_per_mask_amortized=('seconds','median'),
    mean_mask_area_fraction=('mask_area_fraction','mean')).reset_index()
seg_summary.to_csv(RESULTS/'metrics'/'segmentation_comparison.csv',index=False)
display(seg_summary.round(4))
print('Среднее согласие двух методов, не ground-truth IoU:',
      pd.DataFrame(mask_agreement)['iou_between_methods'].mean() if mask_agreement else 'нет парных масок')

    method  n_masks  success_rate  mean_seconds_per_mask_amortized  \
0  GrabCut       11           1.0                           4.6046   
1     SAM2       11           1.0                           3.8517   

   median_seconds_per_mask_amortized  mean_mask_area_fraction  
0                             3.4753                   0.1814  
1                             3.5569                   0.2299  

Среднее согласие двух методов, не ground-truth IoU: 0.760848913336639


## Задание 4. Анализ ошибок и итоговое сравнение (5 баллов)

**Что необходимо просмотреть на сохранённых PNG:**

1. **Ложные обнаружения:** мелкие частицы, артефакты фона и детали внутренних органов, принятые за отдельные COCO-объекты; причину фиксировать по визуальному сравнению.
2. **Пропуски:** прозрачные/маленькие организмы, пересекающиеся контуры, периферийные особи, чья часть за пределами кадра.
3. **Шум/размытие:** сравнить `brightness_mild`, `noise_mild`, `blur_mild`, `flip_horizontal` с original для одного `image_id`; следить за исчезновением тонких отростков.
4. **Неточный box:** SAM 2 не восстанавливает организмы за пределами подсказки гарантированно; GrabCut с неверным box моделирует фон/передний план на неправильной области.
5. **Перекрытие:** GrabCut склонен объединять контактирующие области; SAM 2 тоже может выдать общую маску, если box охватывает несколько организмов. Визуально проверить оба результата, не подменяя это метрикой точности.

Ниже строится автоматическая сводка численных наблюдений. Интерпретацию конкретных случаев редактируют только **после просмотра** сохранённых изображений.

In [14]:
print('Четыре группы детекции:')
display(summary[['experiment','model','n_inferences','mean_detections','zero_detection_fraction',
                 'mean_inference_seconds','match_rate_mean','matched_iou_mean']].round(3))
print('Две модели сегментации:')
display(seg_summary.round(3))

nzero=image_level_df.query("variant == 'original' and count == 0").groupby('model').size().to_dict()
print('Нулевые обнаружения на оригиналах:',nzero)
print('Непосредственная интерпретация: без ручной разметки mAP/Precision/Recall не рассчитываются.')

report = [
 '# Результаты эксперимента',
 f'- Изображений изучено (профиль): {len(profile_df)}',
 f'- Изображений в детекции: {len(selected)}',
 f'- Инференсов детекторов: {len(observations)}',
 f'- Порог confidence: {DET_CONF}; YOLO imgsz: {YOLO_IMGSZ}; MAX_SIDE: {MAX_SIDE}',
 f'- Устройство: {DEVICE}; torch: {torch.__version__}; torchvision: {torchvision.__version__}',
 '',
 '## Четыре эксперимента детекции', summary.round(4).to_markdown(index=False),
 '', '## Два метода сегментации', seg_summary.round(4).to_markdown(index=False),
 '', '## Ограничения',
 '- Нет ground truth; mAP/precision/recall не рассчитаны.',
 '- COCO confidence и количество боксов не интерпретируются как accuracy.',
 '- В случае ручных box-prompts источник отмечен в segmentation_prompts.json.',
 '- Время SAM 2 включает амортизированное время embedding.',
 '- Визуальные ошибки должны быть подтверждены просмотром сохранённых PNG.'
]
(RESULTS/'metrics'/'auto_report.md').write_text('\n'.join(report),encoding='utf-8')

Четыре группы детекции:


   experiment        model  n_inferences  mean_detections  \
0           1         yolo            12            2.917   
1           2         yolo            48            2.688   
2           3  faster_rcnn            12            8.500   
3           4  faster_rcnn            48            9.312   

   zero_detection_fraction  mean_inference_seconds  match_rate_mean  \
0                    0.083                   0.308              NaN   
1                    0.229                   0.307            0.762   
2                    0.000                  13.387              NaN   
3                    0.000                  12.907            0.851   

   matched_iou_mean  
0               NaN  
1             0.912  
2               NaN  
3             0.905  

Две модели сегментации:


    method  n_masks  success_rate  mean_seconds_per_mask_amortized  \
0  GrabCut       11           1.0                            4.605   
1     SAM2       11           1.0                            3.852   

   median_seconds_per_mask_amortized  mean_mask_area_fraction  
0                              3.475                    0.181  
1                              3.557                    0.230  

Нулевые обнаружения на оригиналах: {'yolo': 1}
Непосредственная интерпретация: без ручной разметки mAP/Precision/Recall не рассчитываются.


2382

## Задание 5. Воспроизводимость и экспорт

Все необходимые для проверки артефакты сохранены в `results/`: визуализации аугментаций, изображения с боксами, бинарные маски, overlays, CSV и JSONL. В `run_metadata.json` фиксируются модельные веса, настройки, версии зависимостей и железо. Подготовленный архив содержит вычисленные результаты; **исходные изображения не включаются автоматически**. GitHub должен содержать `ipynb`, `README.md`, `requirements.txt` и папку `results/` после выполнения.

In [15]:
from importlib.metadata import version,PackageNotFoundError
packages=['numpy','pandas','matplotlib','opencv-python-headless','ultralytics','sam2','gdown',
          'scipy','Pillow','huggingface-hub','scikit-image']
versions={}
for name in packages:
    try:versions[name]=version(name)
    except PackageNotFoundError: versions[name]='not installed'
meta={
 'seed':SEED,'device':DEVICE,'python':platform.python_version(),
 'torch':torch.__version__,'torchvision':torchvision.__version__,
 'gpu':torch.cuda.get_device_name(0) if DEVICE=='cuda' else None,
 'model_yolo':'yolo26n.pt','model_faster':'FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT',
 'model_sam':'facebook/sam2.1-hiera-tiny','max_side':MAX_SIDE,'grabcut_max_side':GRABCUT_MAX_SIDE,'yolo_imgsz':YOLO_IMGSZ,
 'conf_threshold':DET_CONF,'max_det':MAX_DET,'n_per_group':N_PER_GROUP,
 'augmentation_variants':AUG_FOR_DETECTION,'selected_detector_for_seg':SEG_DETECTOR,
 'packages':versions,'source_url':FOLDER_URL,
 'work_coordinate_system':'resized image px','original_coordinate_system':'input image px'
}
(RESULTS/'metrics'/'run_metadata.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')

import zipfile
zip_name=Path('plankton_experiment_results.zip')
with zipfile.ZipFile(zip_name,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in RESULTS.rglob('*'):
        if p.is_file():z.write(p,arcname=str(p))
print('Сохранено в',zip_name.resolve(),'Размер (MB):',round(zip_name.stat().st_size/1024**2,2))
print('Теперь включите эти артефакты в репозиторий, если есть разрешение публиковать производные изображения.')

Сохранено в /content/plankton_experiment_results.zip Размер (MB): 116.65
Теперь включите эти артефакты в репозиторий, если есть разрешение публиковать производные изображения.


## Итоговые выводы по фактическому запуску (CPU)

**Условия:** `device=cpu`, PyTorch `2.14.1+cpu`, `seed=42`, `conf=0.10`, `YOLO imgsz=960`, рабочая сторона до 1536 px. Профилировано **86** изображений из частично скачанного набора; для экспериментов детекции выбрано **12** исходных изображений. Выполнено **120** инференсов (12+48 для каждой модели) и получено **713** детекций/записей в `detections.csv`. Метрики относятся к выбранному набору, а не ко всем изображениям исходной папки.

| № | Архитектура и данные | Число инференсов | Обнаружения/снимок | Среднее время (с) |
|---|---|---:|---:|---:|
| 1 | YOLO26n, исходные | 12 | 2,917 | 0,308 |
| 2 | YOLO26n, 4 аугментации | 48 | 2,688 | 0,307 |
| 3 | Faster R-CNN, исходные | 12 | 8,500 | 13,387 |
| 4 | Faster R-CNN, 4 аугментации | 48 | 9,313 | 12,907 |

На исходных изображениях YOLO26n работал **примерно в 43,4 раза быстрее**. На 12 исходных снимках YOLO выдаёт 35 боксов, Faster R-CNN — 102, но у Faster R-CNN много дублирующихся прямоугольников. Например, на `AAKM801(1).jpg` Faster R-CNN выдаёт 13 пересекающихся рамок на одной особи, в то время как YOLO — одну крупную. Это не значит, что YOLO «точнее», поскольку нет GT и её рамки не всегда охватывают все тонкие детали.

**Устойчивость к аугментациям.** Средний match rate боксов YOLO — **0,762**, Faster R-CNN — **0,851**; средний IoU **сопоставленных** боксов — **0,912** и **0,905**. Для YOLO меньше всего совпадений наблюдалось при горизонтальном отражении (**0,670**) и шуме (**0,683**). Для Faster R-CNN наиболее заметен эффект размытия (**0,792**). Match rate не оценивает истинную точность и может быть высоким даже для систематически ошибочных боксов.

**Ошибки детекции.** Модели COCO не знают класса «планктон» и присваивают организмам классы `bird`, `kite`, `cake`, `person`, `toilet` и др. На `20250213131717749.jpg` YOLO выдаёт 0 рамок; Faster R-CNN даёт 5, но часть из них слишком широка. На `20250213140249905.jpg` YOLO выдал 16 рамок, Faster R-CNN — 33, причём часть рамок объединяет несколько организмов и фон. Изменения яркости, шума и размытия дополнительно меняют количество и расположение предсказаний.

**Сегментация.** GrabCut и SAM 2.1 Hiera Tiny использовали **строго идентичные 11 боксов YOLO** на 6 исходных изображениях; все 11 были автоматически получены детектором (`source=detector`), без ручного fallback. Все программные вызовы завершились успешно. Среднее время на маску — **4,605 с** для GrabCut и **3,852 с** для SAM2 (с амортизированным временем embedding); средняя доля маски на снимке — **0,181** и **0,230**. Межметодное IoU: среднее **0,761**, диапазон **0,334–0,951**. Это лишь совпадение масок, не accuracy.

**Визуальные примеры:**

- `20250213140249905.jpg`, `box_id=2` — успешная локализация отдельной особи, близкие маски GrabCut/SAM2, IoU **0,951**. [PNG сравнения](results/segmentation/comparison/20250213140249905_342a772d_2.png).
- `20250213140249905.jpg`, `box_id=0` — слишком большая рамка охватывает несколько организмов; SAM2 захватывает фон, GrabCut выделяет группы силуэтов, IoU **0,380**. [PNG сравнения](results/segmentation/comparison/20250213140249905_342a772d_0.png).
- `Bosmina cf longirostris Sackhalin (24).jpg`, `box_id=1` — SAM2 включает неоднородный фон/текстуры и распадающиеся области, GrabCut формирует связный контур, IoU **0,334**. [PNG сравнения](results/segmentation/comparison/Bosmina_cf_longirostris_Sackhalin_24_46ebd37c_1.png).
- `AAKM880(11).jpg`, `box_id=0` — маски большой особи похожи, IoU **0,920**, но тонкие отростки могут теряться. [PNG сравнения](results/segmentation/comparison/AAKM880_11_d9fa66e4_0.png).

**Общий вывод:** предобученные универсальные модели без дообучения способны давать предварительные области интереса для планктонных снимков, но классификация по COCO некорректна, а рамки часто неполны, дублируются или объединяют особей. Для указанного учебного процесса **YOLO26n удобнее по скорости и меньшему числу дублирующих рамок**, но это не доказательство большей точности. SAM2 иногда лучше сохраняет цельную область, однако при широких подсказках может захватывать фон; GrabCut чувствителен к фону и тонким прозрачным структурам. Без ручной разметки невозможно корректно вычислить mAP, Precision/Recall и IoU с эталоном. Для практического использования необходима дополнительная проверка.

**Источники и воспроизводимость:** все численные значения и исходные предсказания в `results/metrics/`, выполненный ноутбук сохраняет выходы. Датасет скачался частично, устройство CPU. В публичный GitHub изображения и визуальные производные можно загружать только при соответствующем разрешении правообладателя; иначе численные данные и код публикуются отдельно, а визуальный отчёт передаётся преподавателю разрешённым способом.
